# Week 11 作業：ChatGPT API 與 Gradio 聊天機器人

In [ ]:
# 金鑰從 Colab 左側欄的 🔑 Secrets 讀取（不在 Colab 時改讀環境變數），不寫在程式裡
import os

def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

In [1]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### 1. 申請自己的 OpenAI API 金鑰

請先申請自己的 OpenAI API 金鑰(只有建立完成的當下可以複製 API Keys，如果要刪除 API Key，至少要有一個 API Key 才能刪除舊的。)

[`https://platform.openai.com`](https://platform.openai.com)

**程式的基本設定，請自行修改**

* `api_key`: 由 input 讀入的 API Key
* `character`: ChatGPT "人設"
* `description`: App 介紹及 ChatGPT 第一句話
* `model`: 選用模型
  - 更多的model：https://platform.openai.com/docs/models/overview

* `messages（prompt）`：要傳送的句子會詞彙，其實它可以劃分角色。
* `max_tokens`：限制回傳的最大字數，預設會是 128，但有時候會超過字數（被截斷）。
* `temperature`：
    - 隨機文字組合，範圍 0～1（0 表示不隨機，1 表示完全隨機）
    - 預設 0.5，通常不建議調


In [2]:
from getpass import getpass

In [3]:
api_key = get_secret("OPENAI_API_KEY")
# api_key = getpass("請輸入金鑰：")

In [4]:
model = "gpt-4"
# model="gpt-3.5-turbo"

### 2. 使用 OpenAI 的 API

我們先來安裝 `openai` 套件, 還有快速打造 Web App 的 `gradio`。

In [5]:
!pip install openai
!pip install gradio

### 3. 使用 ChatGPT API

首先使用 `openai` 套件。

In [6]:
from openai import OpenAI

並且呼叫API key 來做使用

In [7]:
client = OpenAI(
    # defaults to os.environ.get("OPENAI_API_KEY")
    api_key=api_key,
)

ChatGPT API 的重點是要把之前對話的內容送給 ChatGPT, 然後他就會有個適當的回應!

角色 (`role`) 一共有三種, 分別是:

* `user`: 是使用者本人傳的訊息，所以一般使用者打在輸入匡的就是user這個role傳prompt給OpenAI。
* `system`: 是類似系統設定的「人設」和文字提示，例如跟OpenAI說他是一個工程師，接下來要請他用他的專業寫code；或是跟OpenAI說他是一個厲害的數學家，要請他用他的專業解題。
* `assistan`: 是ChatGPT ，所以當一個訊息的role是assistant，代表這是AI講的話。例如AI回傳時會說role是assistant，還有另一個用途就是跟AI連續對話時，也可以用這個role去實現。

In [8]:
prompt = "我想知道經典的文學電影推薦。"

messages = [{"role":"system", "content":"你是一個對於電影賞析頗具專業的機器人，回應都簡短，儘量不要超過二十個字，回應專業且溫柔。"},
            {"role": "user", "content": prompt}]

檢查messages 的內容是否正確

In [9]:
messages

[{'role': 'system', 'content': '你是一個對於電影賞析頗具專業的機器人，回應都簡短，儘量不要超過二十個字，回應專業且溫柔。'},
 {'role': 'user', 'content': '我想知道經典的文學電影推薦。'}]

把訊息丟給ChatGPT ，看他是否能正常運作。

In [10]:
chat_completion = client.chat.completions.create(
    messages=messages,
    model="gpt-3.5-turbo",
)

In [11]:
chat_completion

ChatCompletion(id='chatcmpl-9Oj09y9FpoOFavTF9NTekPaUlGA5n', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='「麥田捕手」、「苦悶的故事」、「納尼亞傳奇」都不錯。', role='assistant', function_call=None, tool_calls=None))], created=1715679885, model='gpt-3.5-turbo-0125', object='chat.completion', system_fingerprint=None, usage=CompletionUsage(completion_tokens=39, prompt_tokens=104, total_tokens=143))

In [12]:
#只顯示必要的訊息字詞
chat_completion.choices[0].message.content

'「麥田捕手」、「苦悶的故事」、「納尼亞傳奇」都不錯。'

In [13]:
reply = chat_completion.choices[0].message.content

In [14]:
reply

'「麥田捕手」、「苦悶的故事」、「納尼亞傳奇」都不錯。'

將對話塞回messages 讓他可以接續對話！

In [15]:
messages.append({"role":"assistant", "content":reply})

In [16]:
messages

[{'role': 'system', 'content': '你是一個對於電影賞析頗具專業的機器人，回應都簡短，儘量不要超過二十個字，回應專業且溫柔。'},
 {'role': 'user', 'content': '我想知道經典的文學電影推薦。'},
 {'role': 'assistant', 'content': '「麥田捕手」、「苦悶的故事」、「納尼亞傳奇」都不錯。'}]

### 4. 打造一個可以一直說下去的 ChatGPT

In [17]:
icon = "(*ﾟ▽ﾟ*)movie bb: "
messages = [{"role":"system", "content":"你是一個對於電影賞析頗具專業的機器人，回應都簡短，儘量不要超過二十個字，回應專業且溫柔。"}]

while True:
    prompt = input('> ')
    if 'bye' in prompt:
        print('再見, 下次再聊!')
        break
    messages.append({"role": "user", "content": prompt})
    chat_completion = client.chat.completions.create(
        messages=messages,
        model=model,
        )

    reply = chat_completion.choices[0].message.content
    print(icon + reply)
    print()
    messages.append({"role": "assistant", "content": reply})

> bte
(*ﾟ▽ﾟ*)movie bb: 抱歉，您的問題似乎有誤。請詳細說明您需要的電影相關資訊。

> bye
再見, 下次再聊!


### 5. 最後用 `gradio` 打造 ChatGPT 網路應用程式!

In [18]:
import gradio as gr

In [19]:
character = '''你是一個訊息戀愛顧問，專門幫助解答戀愛中的困惑。你需要能夠敏銳地分析訊息，提供建議，並且用溫暖和理解的語氣回答問題。
你的目標是幫助使用者更好地理解對方的訊息和自己的想法，以促進健康和滿意的戀愛關係。'''

description = '''你好！我是你的戀愛顧問，很高興能夠幫助你解決戀愛中訊息聊天中的困惑。不論是對方發來的訊息讓你不甚理解，又或是你不確定自己的回應是否恰當，我都會在這裡幫助你！
我們可以一起分析對方的訊息，並且探討最佳的回應方式，你可以隨時告訴我你的問題和感受。'''

In [20]:
messages = [{"role":"system",
             "content":character},
            {"role":"assistant",
            'content':description}]


In [21]:
def pipi(prompt, history):
    history = history or []
    global messages

    messages.append({"role": "user", "content": prompt})
    chat_completion = client.chat.completions.create(
        messages=messages,
        model=model,
        )
    reply = chat_completion.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    history = history + [[prompt, reply]]

    return history, history

In [22]:
chatbot = gr.Chatbot()

In [23]:
iface = gr.Interface(pipi,
                     inputs=["text", "state"],
                     outputs=[chatbot, "state"],
                     title="訊息戀愛顧問",
                     description=description)

In [ ]:
iface.launch(share=True, debug=True)